# Prepare data for data publishing

Data publishing:
What we need:
	- Simulations: ERA5 and JRA55 simulations
    - Time periods: 1980-2010, and 2023. Thus a total of ~32 years of data

In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
from tqdm import tqdm
import glob
import sys, os

In [ ]:
# ERA5 run:
# -----------------
base = '/g/data/e14/rmh561/access-om2/archive/025deg_era5_iaf_1958cycle1/'
outname = 'ACCESS-OM2-025_ERA5_'

# 1980-2010:
outputs = np.arange(22,53,1)
MLDfile = '/g/data/e14/rmh561/England_NAMHW_project/era5_run/MLD_025deg_era5_iaf_1958cycle1_1980-2010.nc'
SWbasefile = '/g/data/e14/rmh561/England_NAMHW_project/era5_run/SW_base_025deg_era5_iaf_1958cycle1_1980-2010.nc'

# # 2023:
# outputs = [65]
# MLDfile = '/g/data/e14/rmh561/England_NAMHW_project/era5_run/MLD_025deg_era5_iaf_1958cycle1_2023.nc'
# SWbasefile = '/g/data/e14/rmh561/England_NAMHW_project/era5_run/SW_base_025deg_era5_iaf_1958cycle1_2023.nc'

In [ ]:
# JRA55 run:
# -----------------
outname = 'ACCESS-OM2-025_JRA55_'

# 1980-2009:
base = '/g/data/ik11/outputs/access-om2-025/025deg_jra55_iaf_omip2_cycle6/'
outputs = np.arange(327,357)
MLDfile = '/g/data/e14/rmh561/England_NAMHW_project/jra55_run/MLD_025deg_jra55_iaf_omip2_cycle6_1980-2010.nc'
SWbasefile = '/g/data/e14/rmh561/England_NAMHW_project/jra55_run/SW_base_025deg_jra55_iaf_omip2_cycle6_1980-2010.nc'

# 2010:
base = '/g/data/ik11/outputs/access-om2-025/025deg_jra55_iaf_omip2_cycle6/'
outputs = [357]
MLDfile = '/g/data/e14/rmh561/England_NAMHW_project/jra55_run/MLD_025deg_jra55_iaf_omip2_cycle6_2010.nc'
SWbasefile = '/g/data/e14/rmh561/England_NAMHW_project/jra55_run/SW_base_025deg_jra55_iaf_omip2_cycle6_2010.nc'

# # 2023:
# base = '/g/data/ik11/outputs/access-om2-025/025deg_jra55_iaf_omip2_cycle6_jra55v150_extension/'
# outputs = [371]
# MLDfile = '/g/data/e14/rmh561/England_NAMHW_project/jra55_run/MLD_025deg_jra55_iaf_omip2_cycle6_jra55v150_extension_May-Aug2023.nc'
# SWbasefile = '/g/data/e14/rmh561/England_NAMHW_project/jra55_run/SW_base_025deg_jra55_iaf_omip2_cycle6_jra55v150_extension_May-Aug2023.nc'

In [ ]:
variables = {'ocean_month.nc':['temp','salt','lw_heat','sens_heat','evap_heat','swflx','net_sfc_heating']}

output_dir = '/g/data/e14/rmh561/England_NAMHW_project/data_publishing/'

region = [-100, 10, 0, 60]
depths = [0, 20]
rho0 = 1035.
Cp = 3992.10322329649

In [ ]:
# Open extra files:
dsMLD = xr.open_dataset(MLDfile)
dsSWbase = xr.open_dataset(SWbasefile)
dsgrid = xr.open_dataset(base + 'output%03d/ocean/' % outputs[0] + 'ocean_grid.nc')[['geolon_t','geolat_t']].sel(xt_ocean=slice(region[0],region[1]),yt_ocean=slice(region[2],region[3])).load()

# Add attributes:
dsMLD.mld.attrs.update({'long_name':'mixed layer depth computed according to 0.125kgm-3 density criterion','units':'m'})
dsSWbase.SW_base.attrs.update({'long_name':'shortwave penetration below mixed layer depth','units':'Wm-2'})

In [ ]:
for output in tqdm(outputs):
    file = 'ocean_month.nc'
    vars = variables[file]
    
    
    # Extra monthly averaged variables:
    ds = xr.open_dataset(base + 'output%03d/ocean/' % output + file).sel(xt_ocean=slice(region[0],region[1]),yt_ocean=slice(region[2],region[3])).isel(st_ocean=slice(depths[0],depths[1]+1))[vars].load()
    year = int(ds.time[0].dt.year.values)
    # Add MLD and SW penetration:
    
    time_per = slice(str(year) + '-01-01',str(year) + '-12-31')
    ds['mld'] = dsMLD.sel(time=time_per).sel(xt_ocean=slice(region[0],region[1]),yt_ocean=slice(region[2],region[3]))['mld'].load()
    ds['sw_pen'] = dsSWbase.sel(time=time_per).sel(xt_ocean=slice(region[0],region[1]),yt_ocean=slice(region[2],region[3]))['SW_base'].load()
    
    # Add coordinates:
    ds = ds.assign_coords({'geolon_t':dsgrid['geolon_t'].drop_vars(['geolon_t','geolat_t']),'geolat_t':dsgrid['geolat_t'].drop_vars(['geolon_t','geolat_t'])})
    
    # Save to netcdf:
    ds.to_netcdf(output_dir + outname + str(year) + '.nc')

## Just compare SW radiations:

In [ ]:
# JRA-55:
SW_clim = xr.open_dataset('/g/data/e14/rmh561/England_NAMHW_project/SW_surf_025deg_jra55_iaf_omip2_cycle6_1980-2010.nc').SW_surf.load()
SW_2023 = xr.open_dataset('/g/data/e14/rmh561/England_NAMHW_project/SW_surf_025deg_jra55_iaf_omip2_cycle6_jra55v150_extension_May-Aug2023.nc').SW_surf.load()

In [ ]:
SW_anom = SW_2023.groupby('time.month') - SW_clim.groupby('time.month').mean()

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=2,figsize=(12,10))
axs = axes.reshape(-1)
for ti in range(4):
    SW_anom.isel(time=ti).plot(ax=axs[ti],vmin=-60,vmax=60.,cmap='RdBu_r')
for ax in axs:
    ax.set_xlim([-80,10])
    ax.set_ylim([0,75])
    ax.set_facecolor('k')
plt.savefig('JRA55_SWflux_anom.png',dpi=150,bbox_inches='tight')

In [ ]:
# ERA-5:
SW_surfs_era5 = []
outputs = np.arange(0,44)
for output in tqdm(outputs):
    SW_surfs_era5.append(xr.open_dataset(glob.glob('/g/data/ik11/outputs/access-om2-025/025deg_era5_iaf/output%03d/ocean/ocean-2d-swflx-1-monthly-mean-ym*.nc' % output)[0]).swflx)

In [ ]:
SW_surfs_era5 = xr.concat(SW_surfs_era5,dim='time')

In [ ]:
SW_anom_era5 = SW_surfs_era5.sel(time=slice('2023-05-01','2023-09-01')).groupby('time.month') - SW_surfs_era5.sel(time=slice('1980-01-01','2011-01-01')).groupby('time.month').mean()

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=2,figsize=(12,10))
axs = axes.reshape(-1)
for ti in range(4):
    SW_anom_era5.isel(time=ti).plot(ax=axs[ti],vmin=-60,vmax=60.,cmap='RdBu_r')
for ax in axs:
    ax.set_xlim([-80,10])
    ax.set_ylim([0,75])
    ax.set_facecolor('k')
plt.savefig('ERA5_SWflux_anom.png',dpi=150,bbox_inches='tight')

## Compute and save mixed layer depth:

In [ ]:
ds

In [ ]:
%%time 
# Compute mixed layer depth:
# base = '/g/data/ik11/outputs/access-om2-025/025deg_jra55_iaf_omip2_cycle6/'
# outputs = np.arange(327,357)
# out_name = 'MLD_025deg_jra55_iaf_omip2_cycle6_1980-2010.nc'

# base = '/g/data/ik11/outputs/access-om2-025/025deg_jra55_iaf_omip2_cycle6_jra55v150_extension/'
# outputs = [371]
# out_name = 'MLD_025deg_jra55_iaf_omip2_cycle6_jra55v150_extension_May-Aug2023.nc'

#base = '/g/data/e14/rmh561/access-om2/archive/025deg_era5_iaf_1958cycle1/'
#outputs = np.arange(22,53)
#out_name = 'MLD_025deg_era5_iaf_1958cycle1_1980-2010.nc'

outputs = [65]
out_name = 'MLD_025deg_era5_iaf_1958cycle1_2023.nc'

mlds = []
for output in tqdm(outputs):

    base2 = base + 'output%03d/ocean/' % output
    ds_month = xr.open_dataset(base2 + 'ocean_month.nc')
    dht = ds_month.dht.load()
    pot_rho_0 = ds_month.pot_rho_0.load()
    mld = dht.where(pot_rho_0 - pot_rho_0.isel(st_ocean=0) < 0.125).sum('st_ocean') # mixed layer depth
    mlds.append(mld)

In [ ]:
# Concat 
mld_all = xr.concat(mlds,dim='time')

In [ ]:
# Save to file:
mld_all.rename('mld').to_netcdf(tmp_dir + out_name)

In [ ]:
# Compute and save climatologies:
mld_clim = xr.open_dataset(tmp_dir + 'MLD_025deg_era5_iaf_1958cycle1_1980-2010.nc').mld.groupby('time.month').mean()
mld_clim.to_netcdf(tmp_dir + 'MLD_025deg_era5_iaf_1958cycle1_1980-2010_clim.nc')

## Compute and save SW penetration climatology:

## Save surface fluxes within MLD and climatological MLD, and discrete climatological MLD:

In [ ]:
# base = '/g/data/ik11/outputs/access-om2-025/025deg_jra55_iaf_omip2_cycle6/'
# outputs = np.arange(327,357)
# out_name_base = '_025deg_jra55_iaf_omip2_cycle6_1980-2010.nc'

# base = '/g/data/ik11/outputs/access-om2-025/025deg_jra55_iaf_omip2_cycle6_jra55v150_extension/'
# outputs = [371]
# out_name = 'MLD_025deg_jra55_iaf_omip2_cycle6_jra55v150_extension_May-Aug2023.nc'

base = '/g/data/e14/rmh561/access-om2/archive/025deg_era5_iaf_1958cycle1/'
outputs = np.arange(22,53)
out_name_base = '_025deg_era5_iaf_1958cycle1_1980-2010'

# outputs = [65]
# out_name_base = '_025deg_era5_iaf_1958cycle1_2023.nc'

# mld_clim_name = 'MLD_025deg_era5_iaf_1958cycle1_1980-2010_clim.nc'
# mld_clim = xr.open_dataset(tmp_dir + mld_clim_name).load()

SW_heat_clim_name = 'SW_heat_025deg_era5_iaf_1958cycle1_1980-2010_clim.nc'
SW_heat_clim = xr.open_dataset(tmp_dir + SW_heat_clim_name).load()

SW_surfs = []
SW_bases = []
SW_base_clims = []
SW_clim_bases = []
MLD_discretes = []

for output in tqdm(outputs):
    
    base2 = base + 'output%03d/ocean/' % output
    ds_month = xr.open_dataset(base2 + 'ocean_month.nc')
    
    # Load vars:
#    SW_surf = ds_month.swflx.load()
#    sw_heat = ds_month.sw_heat.load()
    dzt = ds_month.dht.load()
    z = dzt.cumsum('st_ocean').load()
    rho = ds_month.pot_rho_0.load()
    
    # SW_base into full MLD:
#    SW_base = sw_heat.where(rho - rho.isel(st_ocean=0) < 0.125).sum('st_ocean').load()

    # SW clim into full MLD:
    SW_clim_base = xr.zeros_like(rho.isel(st_ocean=0)).copy(deep=True)
    for month in range(12):
        SW_clim_base[month,:,:] = SW_heat_clim.sw_heat.isel(time=month).where((rho.isel(time=month) - rho.isel(time=month).isel(st_ocean=0) < 0.125)).sum('st_ocean').load()  
        
    # SW_base into clim MLD:
#    MLDclim_mask = ((z.groupby('time.month')-mld_clim.mld)<0).load()
#    SW_base_clim = sw_heat.groupby('time.month').where(MLDclim_mask).sum('st_ocean').load()
#    MLD_discrete = dzt.groupby('time.month').where(MLDclim_mask).sum('st_ocean').load()

    # Append to lists:
#    SW_surfs.append(SW_surf)
#    SW_bases.append(SW_base)
#    SW_base_clims.append(SW_base_clim)
    SW_clim_bases.append(SW_clim_base)
#    MLD_discretes.append(MLD_discrete)

In [ ]:
# Concat 
#SW_surf = xr.concat(SW_surfs,dim='time')
#SW_base = xr.concat(SW_bases,dim='time')
#SW_base_clim = xr.concat(SW_base_clims,dim='time')
#MLD_discrete = xr.concat(MLD_discretes,dim='time')

In [ ]:
# Output individual files after failed concat:
out_name_base = '_025deg_era5_iaf_1958cycle1_1980-2010.nc' 
for i, output in tqdm(enumerate(outputs)):
#    SW_surfs[i].rename('SW_surf').to_netcdf(tmp_dir + 'SW_surf' + out_name_base)# + '_%03d' % output + '.nc')
#    SW_bases[i].rename('SW_base').to_netcdf(tmp_dir + 'SW_base' + out_name_base)# + '_%03d' % output + '.nc')
#    SW_base_clims[i].rename('SW_base_clim').to_netcdf(tmp_dir + 'SW_base_clim' + out_name_base)# + '_%03d' % output + '.nc')
    SW_clim_bases[i].rename('SW_clim_base').to_netcdf(tmp_dir + 'SW_clim_base' + out_name_base + '_%03d' % output + '.nc')
#    MLD_discretes[i].rename('MLD_discrete').to_netcdf(tmp_dir + 'MLD_discrete' + out_name_base)# + '_%03d' % output + '.nc')

In [ ]:
# Save concatenated version:
SW_surf = xr.open_mfdataset(tmp_dir + 'SW_clim_base_025deg_era5_iaf_1958cycle1_1980-2010*.nc',concat_dim='time',combine='nested',parallel=True,coords='minimal')
SW_surf.load()
SW_surf.to_netcdf(tmp_dir + 'SW_clim_base_era5_iaf_1958cycle1_1980-2010.nc')

In [ ]:
# Save concatenated version:
SW_surf = xr.open_mfdataset(tmp_dir + 'SW_surf_025deg*.nc',concat_dim='time',combine='nested',parallel=True,coords='minimal')
SW_surf.load()
SW_surf.to_netcdf(tmp_dir + 'SW_surf_era5_iaf_1958cycle1_1980-2010.nc')

# Save concatenated version:
SW_surf = xr.open_mfdataset(tmp_dir + 'SW_base_025deg*.nc',concat_dim='time',combine='nested',parallel=True,coords='minimal')
SW_surf.load()
SW_surf.to_netcdf(tmp_dir + 'SW_base_era5_iaf_1958cycle1_1980-2010.nc')

# Save concatenated version:
SW_surf = xr.open_mfdataset(tmp_dir + 'SW_base_clim_025deg*.nc',concat_dim='time',combine='nested',parallel=True,coords='minimal')
SW_surf.load()
SW_surf.to_netcdf(tmp_dir + 'SW_base_clim_era5_iaf_1958cycle1_1980-2010.nc')

# Save concatenated version:
SW_surf = xr.open_mfdataset(tmp_dir + 'MLD_discrete_025deg*.nc',concat_dim='time',combine='nested',parallel=True,coords='minimal')
SW_surf.load()
SW_surf.to_netcdf(tmp_dir + 'MLD_discrete_era5_iaf_1958cycle1_1980-2010.nc')

In [ ]:
# Compute and save climatology:
mld_clim = xr.open_dataset(tmp_dir + 'MLD_discrete_025deg_era5_iaf_1958cycle1_1980-2010.nc').MLD_discrete.groupby('time.month').mean()
mld_clim.to_netcdf(tmp_dir + 'MLD_discrete_025deg_era5_iaf_1958cycle1_1980-2010_clim.nc')